# Unidade 5 — Guia de estudo

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/00_guia_da_unidade.ipynb)
## Como comparar grupos, períodos e documentos?

![Dois conjuntos de documentos passam por distribuições e incerteza; dois textos passam por conjuntos, vetores e alinhamento; ambos os caminhos retornam a documentos inspecionados por uma lupa.](imagens/00_abertura_conceitual.png)

*Ilustração conceitual gerada para a abertura. Os documentos são genéricos e
fictícios; a imagem não constitui evidência histórica.*

**Problema orientador:** como verificar se grupos sociais, instituições,
períodos ou documentos apresentam diferenças relevantes?

A Unidade 4 descreveu padrões e formulou hipóteses exploratórias. Agora
aprenderemos a definir uma comparação, estimar seu tamanho, explicitar a
incerteza quando pertinente e comparar documentos por representações textuais.
Comparar não é apenas subtrair números ou ordenar similaridades: é justificar
unidades, medidas, pressupostos e alcance.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente local: nenhuma clonagem necessária.


A pergunta orientadora se divide em dois percursos que compartilham a mesma
exigência: toda medida precisa retornar aos casos e ao contexto.

## Objetivos e percurso

![Seis etapas ligam pergunta, estimativa, incerteza, representação textual, comparação e interpretação; uma seta retorna às decisões anteriores.](imagens/00_percurso_comparacao.svg)

| Notebook | Pergunta central | Produto |
|---|---|---|
| 00 | O que torna uma comparação defensável? | Escolha preliminar |
| 01 | Qual é o tamanho da diferença? | Quadro de estimativas |
| 02 | Que incerteza acompanha a estimativa? | Ficha inferencial |
| 03 | Como transformar documentos em vetores? | Matriz textual |
| 04 | Em que sentido dois textos são semelhantes? | Relatório de pares |
| 05 | Que comparação sustenta o argumento? | Análise comparativa |

Ao final, você deverá distinguir estimativa, incerteza, valor de *p*, tamanho
de efeito e relevância; construir matriz documento-termo, TF-IDF e três medidas
de comparação textual; e inspecionar qualitativamente os casos.

In [2]:
import json
import pandas as pd

documentos = pd.read_csv("dados/documentos.csv")
proveniencia = json.loads(open("dados/proveniencia.json", encoding="utf-8").read())
print("Registros:", len(documentos))
print("Natureza:", proveniencia["natureza"])
documentos.head(3)

Registros: 24
Natureza: dados inteiramente fictícios


,id_documento,ano,genero,local,tema,palavras,pessoas,paginas,texto
0,D001,1890,editorial,Capital,educação,320,0,1,A escola noturna amplia a instrução pública e ...
1,D002,1891,notícia,Capital,trabalho,367,3,2,O trabalho nas oficinas reúne jornada salário ...
2,D003,1892,carta,Capital,progresso,414,6,3,A estrada e a máquina anunciam progresso e tam...


A inspeção confirma que trabalharemos com os mesmos 24 registros fictícios da
Unidade 4. Essa continuidade permite testar hipóteses já formuladas, mas não
transforma a coleção em amostra probabilística.

## Regra de interpretação

Para toda comparação, registre:

1. unidades, grupos e medida;
2. diferença ou similaridade observada;
3. incerteza ou sensibilidade pertinente;
4. casos que qualificam o agregado;
5. afirmação sustentada e limite.

Intervalos e testes só possuem interpretação inferencial sob um procedimento e
uma população-alvo defensáveis. Em um corpus completo, a diferença descritiva
pode ser o resultado relevante; em uma coleção de conveniência, um pequeno valor
de *p* não repara o mecanismo de seleção.

## Questões críticas

Elas atravessarão todos os experimentos:

1. O que significa considerar dois grupos diferentes?
2. O que significa considerar dois textos semelhantes?
3. Como a escolha da medida altera a conclusão?
4. Uma diferença estatística é necessariamente relevante?

## U05-A01 — Diagnóstico inicial

**Atividade individual — 10 minutos.** Responda em Markdown:

- Quando uma diferença de médias é historicamente importante?
- Um intervalo de confiança contém 95% dos dados?
- O valor de *p* informa a probabilidade de a hipótese nula ser verdadeira?
- Dois textos com as mesmas palavras são necessariamente equivalentes?
- Que hipótese da Unidade 4 você gostaria de comparar?

**Minha resposta:**

- **Importância histórica.** Quando a diferença tem tamanho relevante na escala
  original, se mantém sob escolhas razoáveis de medida e de recorte, se apoia
  em grupos comparáveis e pode ser relacionada a processos documentados pela
  bibliografia. Uma diferença pequena em uma base enorme pode ser estável e,
  ainda assim, irrelevante; uma diferença grande pode resultar só da composição
  dos grupos.
- **Intervalo de confiança.** Não. Ele não descreve onde estão os dados. Ele é
  produzido por um procedimento que, repetido muitas vezes sob suas condições,
  cobriria o parâmetro-alvo em cerca de 95% das repetições.
- **Valor de *p*.** Não. Ele mede quão extremos seriam resultados como o
  observado *se* a hipótese nula e o procedimento fossem verdadeiros. Não dá a
  probabilidade de a hipótese nula ser verdadeira (Wasserstein e Lazar, 2016).
- **Mesmas palavras.** Não. “Trabalho não é progresso” e “progresso não é
  trabalho” têm as mesmas palavras e sentidos opostos. Nos títulos do Goodreads,
  “Angels Flight” e “Angels' Flight” quase coincidem e são livros diferentes,
  de autores diferentes.
- **Hipótese a comparar.** H1 da Unidade 4: a diferença de alcance entre livros
  de autoras e de autores pode estar associada à idade dos livros e ao gênero
  literário, mais do que ao gênero da autoria isoladamente. Também quero
  verificar H2 (maior taxa de resenhas nos livros de autoras).

O diagnóstico torna visíveis pressupostos que serão revistos. O produto final
deverá mostrar uma escolha métrica, uma análise de sensibilidade e retorno aos
casos, não apenas uma saída computacional.

## Produto e avaliação

A análise comparativa será avaliada por alinhamento à pergunta, comparabilidade
dos grupos, correção da medida, transparência dos pressupostos, inspeção de casos,
relevância substantiva, limites e reprodutibilidade.

Siga para o Notebook 01. Começaremos pelo tamanho observado da diferença antes de
introduzir testes ou intervalos.